# Handling Class Imbalance — Breast Cancer Dataset

**Techniques:** Synthetic imbalance creation, `class_weight='balanced'`, SMOTE
**Business context:** In healthcare, the condition of interest (disease, readmission, fraud) is often rare. A naive model predicts "healthy" for everything and achieves high accuracy but fails where it matters most.

This notebook demonstrates the problem and two practical solutions.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, ConfusionMatrixDisplay
from collections import Counter

try:
    from imblearn.over_sampling import SMOTE
    SMOTE_AVAILABLE = True
    print("imbalanced-learn loaded -- SMOTE demo enabled.")
except ImportError:
    SMOTE_AVAILABLE = False
    print("imbalanced-learn not installed. Run: pip install imbalanced-learn")
    print("Proceeding with class_weight demo only.")

## 1. Create a Synthetic Imbalanced Dataset

We artificially create a **10:1 class imbalance** to simulate a realistic scenario (e.g., rare disease detection or low readmission rate).

In [ ]:
raw = load_breast_cancer(as_frame=True)
X_all, y_all = raw.data.values, raw.target.values

# Keep all benign (majority) but only 35 malignant (minority) -> ~10:1 ratio
np.random.seed(42)
malignant_idx = np.where(y_all == 0)[0]
benign_idx    = np.where(y_all == 1)[0]
malignant_keep = np.random.choice(malignant_idx, size=35, replace=False)
keep = np.concatenate([benign_idx, malignant_keep])

X_imb, y_imb = X_all[keep], y_all[keep]
print(f"Imbalanced distribution: {Counter(y_imb)}")
print(f"Ratio: {Counter(y_imb)[1]}:{Counter(y_imb)[0]} "
      f"({Counter(y_imb)[1] / Counter(y_imb)[0]:.1f}:1 benign:malignant)")

## 2. Baseline: Default Logistic Regression (no weighting)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_imb, y_imb, test_size=0.2, random_state=42, stratify=y_imb
)
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

lr_default = LogisticRegression(max_iter=1000, random_state=42)
lr_default.fit(X_train_sc, y_train)
y_pred_default = lr_default.predict(X_test_sc)

print("=== Default (no class weighting) ===")
print(classification_report(y_test, y_pred_default, target_names=raw.target_names))

## 3. Solution 1 — `class_weight='balanced'`

Sklearn automatically adjusts the loss function to penalize misclassifying the minority class more heavily. Simple to apply — just one parameter change.

In [ ]:
lr_balanced = LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced')
lr_balanced.fit(X_train_sc, y_train)
y_pred_balanced = lr_balanced.predict(X_test_sc)

print("=== class_weight='balanced' ===")
print(classification_report(y_test, y_pred_balanced, target_names=raw.target_names))

## 4. Solution 2 — SMOTE

SMOTE (Synthetic Minority Over-sampling Technique) generates new synthetic samples for the minority class by interpolating between existing minority points. This is applied only to training data — never to test data.

In [ ]:
if SMOTE_AVAILABLE:
    smote = SMOTE(random_state=42)
    X_train_res, y_train_res = smote.fit_resample(X_train_sc, y_train)
    print(f"Before SMOTE: {Counter(y_train)}")
    print(f"After SMOTE:  {Counter(y_train_res)}")

    lr_smote = LogisticRegression(max_iter=1000, random_state=42)
    lr_smote.fit(X_train_res, y_train_res)
    y_pred_smote = lr_smote.predict(X_test_sc)

    print("\n=== SMOTE + Logistic Regression ===")
    print(classification_report(y_test, y_pred_smote, target_names=raw.target_names))
else:
    print("Install imbalanced-learn to run this cell: pip install imbalanced-learn")

## 5. Side-by-Side Confusion Matrices

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, (name, preds) in zip(axes, [
    ('Default (no weighting)', y_pred_default),
    ("class_weight='balanced'", y_pred_balanced),
]):
    ConfusionMatrixDisplay.from_predictions(
        y_test, preds, display_labels=raw.target_names,
        colorbar=False, ax=ax
    )
    ax.set_title(name)

plt.suptitle('Impact of Class Weighting — Imbalanced Healthcare Data (10:1)',
             fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

## Key Takeaways

| Approach | Malignant Recall | Notes |
|---|---|---|
| Default LR | Low (misses sick patients) | High accuracy is misleading |
| `class_weight='balanced'` | High | Best first fix — zero extra effort |
| SMOTE | High | More control; cross-validate carefully |

**In healthcare, false negatives (missing a diagnosis) are often more costly than false positives. Always evaluate with recall and F1, not just accuracy.**